In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [ ]:
# 1) LOAD DATA
movies  = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags    = pd.read_csv("tags.csv")

print(" Data Loaded Successfully")
print("Movies :", movies.shape)
print("Ratings:", ratings.shape)
print("Tags   :", tags.shape)

✅ Data Loaded Successfully
Movies : (9742, 3)
Ratings: (100836, 4)
Tags   : (3683, 4)


In [ ]:
# 2) BASIC EDA

print("\n=== BASIC EDA ===")
print("Unique Users :", ratings["userId"].nunique())
print("Unique Movies:", ratings["movieId"].nunique())
print("Rating Range :", ratings["rating"].min(), "–", ratings["rating"].max())
print("\nRatings Distribution:")
print(ratings["rating"].value_counts().sort_index())


=== BASIC EDA ===
Unique Users : 610
Unique Movies: 9724
Rating Range : 0.5 – 5.0

Ratings Distribution:
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


In [ ]:
# 3) CLEANING

movies.drop_duplicates(inplace=True)
ratings.drop_duplicates(inplace=True)
tags.drop_duplicates(inplace=True)

movies.fillna("", inplace=True)
ratings.dropna(inplace=True)
tags.fillna("", inplace=True)

ratings["userId"]  = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)
ratings["rating"]  = ratings["rating"].astype(float)
movies["movieId"]  = movies["movieId"].astype(int)

print("\nCleaning Done — no nulls in ratings, movies, tags.")


✅ Cleaning Done — no nulls in ratings, movies, tags.


In [ ]:
# 4) CONTENT-BASED FILTERING

movie_tags = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)
movie_tags.columns = ["movieId", "tags_text"]

movies = movies.merge(movie_tags, on="movieId", how="left")
movies["tags_text"] = movies["tags_text"].fillna("")

movies["content"] = (
    movies["title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["tags_text"]
)

tfidf        = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["content"])
cosine_sim   = cosine_similarity(tfidf_matrix, tfidf_matrix)

movieId_to_index = pd.Series(movies.index, index=movies["movieId"]).drop_duplicates()

RATING_SCALE = (0.5, 5.0)


def content_predict(user_id, movie_id):
    """
    Predict rating using Content-Based Filtering
    based on similarity with movies the user already rated.
    """
    user_movies = ratings[ratings["userId"] == user_id]

    if user_movies.empty:
        return 0.0

    if movie_id not in movieId_to_index:
        return float(user_movies["rating"].mean())

    target_idx = movieId_to_index[movie_id]

    weighted_scores = []
    sim_values = []

    for _, row in user_movies.iterrows():
        rated_id = row["movieId"]

        if rated_id not in movieId_to_index:
            continue

        rated_idx = movieId_to_index[rated_id]
        sim = cosine_sim[target_idx][rated_idx]

        weighted_scores.append(sim * row["rating"])
        sim_values.append(sim)

    total_sim = sum(sim_values)

    if total_sim == 0:
        return float(user_movies["rating"].mean())

    return sum(weighted_scores) / total_sim

In [ ]:
# 5) COLLABORATIVE FILTERING (SVD)
reader = Reader(rating_scale=RATING_SCALE)
data   = Dataset.load_from_df(ratings[["userId", "movieId", "rating"]], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

svd_predictions = svd.test(testset)

print("\n=== Collaborative Filtering (SVD) Performance ===")
accuracy.rmse(svd_predictions)
accuracy.mae(svd_predictions)


=== Collaborative Filtering (SVD) Performance ===
RMSE: 0.8807
MAE:  0.6766


0.6765729095860605

In [ ]:
# 6) HYBRID PREDICTION

def _clip(val):
    return float(np.clip(val, RATING_SCALE[0], RATING_SCALE[1]))


def hybrid_predict(user_id, movie_id, alpha=0.4):
    """
    Hybrid prediction = alpha * content + (1-alpha) * collaborative
    """
    beta = 1.0 - alpha
    collab_pred  = _clip(svd.predict(user_id, movie_id).est)
    content_pred = _clip(content_predict(user_id, movie_id))
    return alpha * content_pred + beta * collab_pred

In [ ]:
# 7) EVALUATION FUNCTION
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    threshold = 3.5
    tp = fp = fn = 0

    for a, p in zip(y_true, y_pred):
        if p >= threshold and a >= threshold:
            tp += 1
        elif p >= threshold and a < threshold:
            fp += 1
        elif p < threshold and a >= threshold:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    print(f"\n=== {name} Evaluation ===")
    print(f"RMSE      : {rmse:.4f}")
    print(f"MAE       : {mae:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    return dict(name=name, rmse=rmse, mae=mae,
                precision=precision, recall=recall, f1=f1)

In [9]:
# 8) EVALUATION (ALL MODELS)
y_true, y_collab, y_content, y_hybrid = [], [], [], []

for pred in svd_predictions:
    user   = int(pred.uid)
    movie  = int(pred.iid)
    actual = pred.r_ui

    y_true.append(actual)
    y_collab.append(_clip(pred.est))
    y_content.append(_clip(content_predict(user, movie)))
    y_hybrid.append(hybrid_predict(user, movie, alpha=0.4))

results = [
    evaluate(y_true, y_content, "Content-Based"),
    evaluate(y_true, y_collab,  "Collaborative (SVD)"),
    evaluate(y_true, y_hybrid,  "Hybrid"),
]

print("\n=== Summary Table ===")
print(pd.DataFrame(results).set_index("name").to_string())


=== Content-Based Evaluation ===
RMSE      : 0.7663
MAE       : 0.5898
Precision : 0.8702
Recall    : 0.7101
F1 Score  : 0.7820

=== Collaborative (SVD) Evaluation ===
RMSE      : 0.8807
MAE       : 0.6766
Precision : 0.7947
Recall    : 0.6807
F1 Score  : 0.7333

=== Hybrid Evaluation ===
RMSE      : 0.8122
MAE       : 0.6270
Precision : 0.8270
Recall    : 0.7007
F1 Score  : 0.7586

=== Summary Table ===
                         rmse       mae  precision    recall        f1
name                                                                  
Content-Based        0.766298  0.589787   0.870158  0.710084  0.782013
Collaborative (SVD)  0.880746  0.676573   0.794706  0.680670  0.733281
Hybrid               0.812221  0.626973   0.826988  0.700658  0.758599


In [10]:
# 9) RECOMMENDATION FUNCTION
def recommend_movies(user_id, top_n=10, alpha=0.4):
    all_movie_ids  = movies["movieId"].unique()
    rated_movies   = set(ratings[ratings["userId"] == user_id]["movieId"].values)
    unrated_movies = [m for m in all_movie_ids if m not in rated_movies]

    preds = [(movie_id, hybrid_predict(user_id, movie_id, alpha=alpha)) for movie_id in unrated_movies]
    preds.sort(key=lambda x: x[1], reverse=True)

    result = pd.DataFrame(preds[:top_n], columns=["movieId", "hybrid_score"])
    result = result.merge(movies[["movieId", "title", "genres"]], on="movieId")
    result["hybrid_score"] = result["hybrid_score"].round(3)

    return result[["title", "genres", "hybrid_score"]]

In [11]:
# 10) DEMO

sample_user = int(ratings["userId"].iloc[0])
print(f"\n=== Top Hybrid Recommendations for User {sample_user} ===")
print(recommend_movies(sample_user, top_n=10))


=== Top Hybrid Recommendations for User 1 ===
                                            title         genres  hybrid_score
0                       Dancer in the Dark (2000)  Drama|Musical         4.866
1                Streetcar Named Desire, A (1951)          Drama         4.856
2                             Hustler, The (1961)          Drama         4.818
3                             Pianist, The (2002)      Drama|War         4.815
4                           Cool Hand Luke (1967)          Drama         4.811
5             Guess Who's Coming to Dinner (1967)          Drama         4.801
6                              Raging Bull (1980)          Drama         4.799
7                       King's Speech, The (2010)          Drama         4.794
8  Cinema Paradiso (Nuovo cinema Paradiso) (1989)          Drama         4.794
9                               Casablanca (1942)  Drama|Romance         4.794
